# qdmpy_core ODMR Processors Tutorial

This tutorial demonstrates qdmpy_core's modular processor framework for preparing ODMR data
for spectral fitting.

Topics covered:
1. Load ODMR data with `MatlabLoader` + `ODMRData`
2. Set up a processor pipeline via `ODMR` + `ODMRProcessorManager`
3. Apply processors step by step and visualise their effect
4. Preview fluorescence correction factors
5. Best-practice pipeline for production use

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)

from qdmpy_core.odmr import ODMRData, ODMR
from qdmpy_core.odmr.io import MatlabLoader
from qdmpy_core.odmr.processors import (
    BinningProcessor,
    FluorescenceCorrectionProcessor,
    NormalizationProcessor,
    ODMRProcessorManager,
    OutlierProcessor,
    analyze_fluorescence_effects,
    preview_fluorescence_correction,
)

## 1. Load ODMR data

`MatlabLoader` reads the two `run_*.mat` files (one per field polarity).  
`ODMRData.from_loader()` returns an `ODMRData` whose `.data` attribute is a validated
5-D `xr.DataArray` with named dimensions `(polarity, freq_range, y, x, freq_idx)`.

In [ ]:
DATA_FOLDER = Path.home() / "git" / "qdmpy_core" / "tests" / "data" / "MIL2_FOV1"

loader = MatlabLoader(data_folder=str(DATA_FOLDER))
odmr_data = ODMRData.from_loader(loader)

print(odmr_data.data)
print(f"\nScan dimensions: {odmr_data.scan_dimensions}")
print(f"Frequencies (GHz): {odmr_data.frequencies.shape}")

## 2. Raw spectrum inspection

Plot the mean spectrum over all pixels to understand the raw signal quality.

In [ ]:
freq_ghz = odmr_data.data.coords["freq_ghz"].values
pols = list(odmr_data.data.coords["polarity"].values)
franges = list(odmr_data.data.coords["freq_range"].values)

mean_spectra = odmr_data.data.mean(dim=["y", "x"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for i_fr, frange in enumerate(franges):
    ax = axes[i_fr]
    for pol in pols:
        spectrum = mean_spectra.sel(polarity=pol, freq_range=frange).values
        ax.plot(freq_ghz[i_fr], spectrum, "o-", ms=3, label=pol)
    ax.set_xlabel("Frequency (GHz)")
    ax.set_ylabel("Fluorescence (counts)")
    ax.set_title(f"Mean spectrum — {frange} range")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Raw ODMR spectra (mean over all pixels)", y=1.01)
plt.tight_layout()
plt.show()

## 3. Processor pipeline — step by step

The `ODMR` class wraps an `ODMRData` and exposes `processor_manager` (an
`ODMRProcessorManager`).  Calling `odmr.process_data()` runs the pipeline in order,
returning a new `ODMRData` available as `odmr.processed_data`.

### Step 1 — Outlier masking

`OutlierProcessor` replaces values whose z-score along the frequency axis exceeds
`z_score_threshold` with NaN.  A conservative threshold of 0.003 (0.3 %) avoids
removing real ODMR signal.

In [ ]:
odmr = ODMR(odmr_data)
odmr.processor_manager.add_processor(OutlierProcessor(z_score_threshold=0.003))
odmr.process_data()

proc1 = odmr.processed_data

# Compare mean spectrum before/after outlier removal (centre pixel, low range, neg pol)
cy, cx = odmr_data.scan_dimensions[0] // 2, odmr_data.scan_dimensions[1] // 2

raw_spec = odmr_data.data.sel(polarity="neg", freq_range="low").isel(y=cy, x=cx).values
clean_spec = proc1.data.sel(polarity="neg", freq_range="low").isel(y=cy, x=cx).values

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq_ghz[0], raw_spec, "b.", ms=4, alpha=0.7, label="Raw")
ax.plot(freq_ghz[0], clean_spec, "r-", lw=2, alpha=0.8, label="After outlier mask")
ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("Fluorescence")
ax.set_title("Outlier masking — centre pixel, neg, low range")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Step 2 — Fluorescence correction

Spatial variation of the diamond fluorescence distorts the apparent contrast.  
`FluorescenceCorrectionProcessor` subtracts a scaled version of the spatially-varying
baseline from each pixel's spectrum.

Use `analyze_fluorescence_effects()` and `preview_fluorescence_correction()` to choose
the `correction_factor` before processing.

In [ ]:
# Analyse fluorescence effects — returns automatically selected representative pixel
pixel_idx, baseline_corrected = analyze_fluorescence_effects(odmr_data)

In [ ]:
# Preview different correction factors to pick the best one
for factor in [0.0, 0.2, 0.5]:
    preview_fluorescence_correction(odmr_data, correction_factor=factor, pixel_idx=pixel_idx)

### Step 3 — Normalisation

`NormalizationProcessor(method="max")` divides each pixel spectrum by its maximum value,
so the baseline becomes 1 and dip depths represent contrast (0–1).

### Step 4 — Spatial binning

`BinningProcessor(bin_factor=4)` averages 4×4 pixel blocks, reducing the image from
1920×1200 to 480×300.  This increases SNR at the cost of spatial resolution.

**Order matters:** bin after correction so that the fluorescence correction operates
at full resolution.

## 4. Full recommended pipeline

In [ ]:
odmr = ODMR(odmr_data)

odmr.processor_manager.add_processor(OutlierProcessor(z_score_threshold=0.003))
odmr.processor_manager.add_processor(FluorescenceCorrectionProcessor(correction_factor=0.2))
odmr.processor_manager.add_processor(NormalizationProcessor(method="max"))
odmr.processor_manager.add_processor(BinningProcessor(bin_factor=4))

odmr.process_data()

proc = odmr.processed_data
print(f"Raw shape:       {odmr_data.data.shape}")
print(f"Processed shape: {proc.data.shape}")

## 5. Visualise processed spectra

In [ ]:
freq_ghz_proc = proc.data.coords["freq_ghz"].values
pols_proc = list(proc.data.coords["polarity"].values)
franges_proc = list(proc.data.coords["freq_range"].values)

mean_proc = proc.data.mean(dim=["y", "x"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for i_fr, frange in enumerate(franges_proc):
    ax = axes[i_fr]
    for pol in pols_proc:
        spectrum = mean_proc.sel(polarity=pol, freq_range=frange).values
        ax.plot(freq_ghz_proc[i_fr], spectrum, "o-", ms=3, label=pol)
    ax.set_xlabel("Frequency (GHz)")
    ax.set_ylabel("Normalised intensity")
    ax.set_title(f"Processed — {frange} range")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Processed ODMR spectra (mean over all pixels)", y=1.01)
plt.tight_layout()
plt.show()

## 6. Contrast map

A quick proxy for ODMR signal strength: `max − min` along the frequency axis per pixel.

In [ ]:
contrast = (proc.data.max(dim="freq_idx") - proc.data.min(dim="freq_idx"))
# Average over polarities and ranges for an overall contrast image
contrast_mean = contrast.mean(dim=["polarity", "freq_range"])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(contrast_mean.values, cmap="hot")
ax.set_title("ODMR contrast map (processed)")
ax.set_xlabel("x (binned pixels)")
ax.set_ylabel("y (binned pixels)")
fig.colorbar(im, ax=ax, shrink=0.8, label="max − min")
plt.tight_layout()
plt.show()

## 7. Pipeline serialisation

`ODMRProcessorManager` can export its configuration to a list of dicts and re-create
itself from that config — useful for reproducible analysis scripts.

In [ ]:
config = odmr.processor_manager.pipeline_config
print("Pipeline config:")
for step in config:
    print(f"  {step}")

# Reconstruct an identical manager from the config
restored = ODMRProcessorManager.from_config(config)
print(f"\nRestored processors: {restored.list_processors()}")

## Summary

| Processor | When to use |
|-----------|------------|
| `OutlierProcessor(z_score_threshold=0.003)` | Always — remove sensor glitches |
| `FluorescenceCorrectionProcessor(correction_factor=0.2)` | When spatial fluorescence gradient is visible |
| `NormalizationProcessor(method="max")` | Always before fitting — normalise baseline to 1 |
| `BinningProcessor(bin_factor=4)` | When SNR is low or speed matters |

**Recommended order**: outlier → fluorescence → normalisation → binning

The processed `ODMRData` from `odmr.processed_data` is ready to pass directly to
`FitManager.fit(proc.data, proc.frequencies)`.  See `tutorial.ipynb` for the
complete fitting workflow.